# Multi-Source Sequence Fetch

![Multi-Source Sequence Fetch](https://proto-bio.github.io/proto-assets/images/tool/sequence_fetch/hero.png)

This notebook demonstrates `run_sequence_fetch`, a thin orchestrator that retrieves biological sequences and structures from multiple public databases in a single call. It chains NCBI, UniProt, and PDB lookups with molecule-type routing and cross-fetcher ID sharing, so a UniProt ID resolved during protein retrieval is automatically reused when looking up a PDB structure. The notebook covers fetching a protein sequence by name, combining multiple molecule types in one request, processing batch queries across several targets, and bypassing name resolution by providing known accession identifiers directly.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("sequence_fetch")
display_overview("sequence_fetch")
display_docs_section("sequence_fetch", "Background")

# Unified Sequence Fetch

The `sequence-fetch` tool is a multi-source orchestrator that resolves a batch of heterogeneous sequence and structure requests into a uniform result. Each request names a gene, protein, or RNA target and the molecule types to retrieve (protein, genomic DNA, coding DNA, transcript RNA, inferred pre-mRNA, or PDB structure), optionally pinned by accession overrides. It federates over [NCBI Entrez](https://www.ncbi.nlm.nih.gov/search/), [UniProt](https://www.uniprot.org/), and [RCSB PDB](https://www.rcsb.org/), returning per-request sequences, structure metadata, resolved identifiers, and status.

This tool wraps three public databases rather than a single predictive model, so it has no single primary paper. The underlying sources are [GenBank](https://www.ncbi.nlm.nih.gov/genbank/) ([Sayers et al., 2022](https://doi.org/10.1093/nar/gkab1135)), [UniProt](https://www.uniprot.org/) ([The UniProt Consortium, 2025](https://doi.org/10.1093/nar/gkae1010)), and the [RCSB Protein Data Bank](https://www.rcsb.org/) ([Berman et al., 2000](https://doi.org/10.1093/nar/28.1.235)). Internally, `SequenceFetchInput` wraps a `list[SequenceFetchRequest]`, and each request is resolved independently by molecule type with deterministic, priority-based routing where provided identifiers are consulted before a name-and-organism search. For a protein request the routing priority is a supplied UniProt accession (resolved at `rest.uniprot.org`), then an NCBI protein accession, then a linked PDB entry's FASTA chains, and finally a name-and-organism search. Nucleotide requests use the [NCBI E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25501/) at `https://eutils.ncbi.nlm.nih.gov/entrez/eutils` (esearch, esummary, efetch), with a gene-locus coordinate fallback that fetches the genomic interval directly from the chromosome accession. Structure requests resolve a PDB identifier and read entry metadata from `https://data.rcsb.org`, with chain sequences pulled from `https://www.rcsb.org/fasta/entry`. Every returned record carries a source URL and a SHA256 checksum for provenance, and the NCBI API key and contact email are sanitized out of provenance URLs. Genomic coordinates are interpreted as 1-indexed, inclusive intervals to match biological residue selection conventions. Results reflect the live databases at query time rather than a fixed release snapshot.

## Available tools

In [2]:
display_available_tools("sequence_fetch")

- **`run_sequence_fetch()`** — Fetch DNA, RNA, protein, and structure records from NCBI, UniProt, and PDB

### `run_sequence_fetch`

`run_sequence_fetch` accepts a list of `SequenceFetchRequest` objects, each specifying a target gene or protein name, an organism for disambiguation, and the desired molecule types. It searches UniProt, NCBI, and PDB in sequence, sharing resolved IDs across fetchers so each database is queried at most once per request. Results are returned as a `SequenceFetchOutput` with one `SequenceFetchResult` per request, containing the fetched sequences, structures, resolved IDs, and any per-type errors.

In [3]:
from proto_tools import SequenceFetchInput, SequenceFetchConfig, SequenceFetchRequest, run_sequence_fetch

In [4]:
# Display docs
display_api_reference("sequence_fetch", "input", "run_sequence_fetch")

# Fetch the lac repressor protein sequence from Escherichia coli
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        )
    ]
)

**Input** — `SequenceFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>requests</code> | <code>list[SequenceFetchRequest]</code> | required | One or more retrieval requests |

In [5]:
# Display config docs
display_api_reference("sequence_fetch", "config", "run_sequence_fetch")

# Use default config | see docs above for all fields
config = SequenceFetchConfig()

**Config** — `SequenceFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>max_candidates_per_source</code> | <code>int</code> | <code>5</code> | Maximum database candidates to evaluate per name-based search |
| <code>type_check_mode</code> | <code>Literal['off', 'warn', 'error']</code> | <code>'error'</code> | Molecule-type mismatch handling: 'off' (skip), 'warn' (log + continue), 'error' (fail) |
| <code>ncbi_api_key</code> | <code>str &#124; None</code> | <code>None</code> | Optional NCBI API key (3 to 10 req/s). Defaults to the NCBI_API_KEY env var if not set. |
| <code>ncbi_email</code> | <code>str &#124; None</code> | <code>None</code> | Optional contact email for NCBI. Defaults to the NCBI_EMAIL env var if not set. |

In [6]:
# Run the fetch
output = run_sequence_fetch(inputs, config)

In [7]:
# Display output docs
display_api_reference("sequence_fetch", "output", "run_sequence_fetch")

# Inspect the fetched protein sequence
result = output.results[0]
print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Type: {seq.sequence_type}")
print(f"Source: {seq.source_database} ({seq.accession})")
print(f"Length: {seq.length}")
print(f"Preview: {seq.sequence[:60]}...")

**Output** — `SequenceFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceFetchResult]</code> | <code>[]</code> | Per-request retrieval outcomes |

**`SequenceFetchResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>request_id</code> | <code>str</code> | required | Request identifier |
| <code>target_name</code> | <code>str</code> | required | Original target name |
| <code>organism</code> | <code>str</code> | required | Original organism name |
| <code>requested_types</code> | <code>list[str]</code> | required | Requested molecule types |
| <code>status</code> | <code>Literal['success', 'warning', 'failed']</code> | required | Result status |
| <code>fetched_sequences</code> | <code>list[FetchedSequence]</code> | <code>[]</code> | Retrieved sequence records |
| <code>fetched_structures</code> | <code>list[FetchedStructure]</code> | <code>[]</code> | Retrieved structure records |
| <code>resolved_ids</code> | <code>dict[str, str]</code> | <code>{}</code> | Resolved identifiers used in retrieval |

**`FetchedSequence`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_type</code> | <code>Literal['protein', 'dna_genomic', 'dna_cds', 'rna_transcript', 'rna_premrna']</code> | required | Requested type for this sequence |
| <code>source_database</code> | <code>Literal['ncbi', 'uniprot', 'pdb']</code> | required | Source database for this sequence |
| <code>accession</code> | <code>str &#124; None</code> | <code>None</code> | Source accession identifier |
| <code>sequence</code> | <code>str</code> | required | Retrieved sequence |
| <code>length</code> | <code>int</code> | required | Sequence length |
| <code>checksum_sha256</code> | <code>str &#124; None</code> | <code>None</code> | SHA256 checksum |
| <code>source_url</code> | <code>str &#124; None</code> | <code>None</code> | Source URL for provenance |
| <code>inferred</code> | <code>bool</code> | <code>False</code> | True if sequence was reconstructed (e.g. translated CDS) rather than fetched directly |

**`FetchedStructure`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_id</code> | <code>str</code> | required | PDB accession |
| <code>source_database</code> | <code>Literal['pdb']</code> | <code>'pdb'</code> | Source database |
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Structure title |
| <code>method</code> | <code>str &#124; None</code> | <code>None</code> | PDB experimental method (e.g. X-RAY DIFFRACTION, SOLUTION NMR, ELECTRON MICROSCOPY) |
| <code>resolution</code> | <code>float &#124; None</code> | <code>None</code> | Resolution in Å (None for NMR or fiber diffraction) |
| <code>source_url</code> | <code>str</code> | required | Canonical structure URL |

Target: lacI (Escherichia coli)
Status: success
Resolved IDs: {'uniprot_id': 'P03023'}

Type: protein
Source: uniprot (P03023)
Length: 360
Preview: MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPNRVAQQLAGKQ...


#### Multi-Type Request

A single request can fetch multiple molecule types at once. Here we retrieve the protein sequence, coding DNA sequence, and an experimentally determined structure for TP53. The orchestrator resolves IDs once and shares them across fetchers, so a UniProt ID found during protein retrieval is reused when looking up the associated PDB structure.

In [8]:
# Fetch protein, CDS, and structure for TP53 in a single request
multi_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "dna_cds", "structure"],
            refseq_accession="NM_000546.6",  # pin canonical TP53 mRNA for the correct CDS
        )
    ]
)

multi_output = run_sequence_fetch(multi_inputs, config)
result = multi_output.results[0]

print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

for seq in result.fetched_sequences:
    preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
    print(f"  {seq.sequence_type}: {seq.source_database} ({seq.accession}), length={seq.length}")
    print(f"    {preview}")

for struct in result.fetched_structures:
    res_str = f"{struct.resolution:.1f} A" if struct.resolution else "N/A"
    print(f"  structure: {struct.pdb_id}, {struct.method}, {res_str}")
    print(f"    {struct.title}")

Target: TP53 (Homo sapiens)
Status: warning
Resolved IDs: {'cds_accession': 'NM_000546.6_cds_NP_000537.3_1', 'pdb_id': '1A1U', 'uniprot_id': 'P04637'}

  dna_cds: ncbi (NM_000546.6_cds_NP_000537.3_1), length=1182
    ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGA...
  structure: 1A1U, SOLUTION NMR, N/A
    SOLUTION STRUCTURE DETERMINATION OF A P53 MUTANT DIMERIZATION DOMAIN, NMR, MINIMIZED AVERAGE STRUCTURE


#### Batch Requests

Multiple targets can be processed in a single call by including multiple `SequenceFetchRequest` objects. Each request is resolved independently against the upstream databases.

In [9]:
# Fetch three proteins in one call
batch_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="insulin",
            organism="Homo sapiens",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="GFP",
            organism="Aequorea victoria",
            sequence_types=["protein"],
        ),
    ]
)

batch_output = run_sequence_fetch(batch_inputs, config)

print(f"Completed: {batch_output.num_completed}/{batch_output.num_requests}")
print()

for result in batch_output.results:
    if result.fetched_sequences:
        seq = result.fetched_sequences[0]
        print(f"{result.target_name}: {seq.source_database} ({seq.accession}), length={seq.length}")
    else:
        print(f"{result.target_name}: {result.status} -- {result.errors}")

Completed: 3/3

lacI: uniprot (P03023), length=360
insulin: uniprot (P01308), length=110
GFP: uniprot (P42212), length=238


#### Direct ID Override

When you already have a UniProt accession or PDB ID, you can provide it directly to skip the name search step. This is faster and avoids any ambiguity from name resolution.

In [10]:
# Use known UniProt and PDB IDs to skip name resolution
direct_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "structure"],
            uniprot_id="P04637",
            pdb_id="1TSR",
        )
    ]
)

direct_output = run_sequence_fetch(direct_inputs, config)
result = direct_output.results[0]

print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Protein: {seq.accession}, length={seq.length}")

struct = result.fetched_structures[0]
print(f"Structure: {struct.pdb_id}, {struct.method}")

Status: success
Resolved IDs: {'uniprot_id': 'P04637', 'pdb_id': '1TSR'}

Protein: P04637, length=393
Structure: 1TSR, X-RAY DIFFRACTION


## Export Results

Fetched sequences can be saved to FASTA or JSON format for downstream analysis. The FASTA export includes structured headers with target name, sequence type, source database, and accession for each record.

In [11]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("sequence_fetch_results", export_path=output_dir, file_format="fasta")
print(f"Exported to {output_dir / 'sequence_fetch_results.fasta'}")

output.export("sequence_fetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'sequence_fetch_results.json'}")

Exported to example_output/sequence_fetch_results.fasta
Exported to example_output/sequence_fetch_results.json
